# `irkit` — Comprehensive Implementation Tutorial

Welcome to the deep-dive tutorial for **`irkit`**, a pluggable, distributed hybrid information retrieval toolkit. This notebook functions as a living laboratory where you can learn the concepts, implement the code, and verify it with tests.

### Learning Roadmap Overview
1. **Core Architecture**: Interfaces and Abstract Base Classes.
2. **Keyword Search**: Implementing BM25 Rankers.
3. **Semantic Search**: Vector embeddings and FAISS.
4. **Hybrid Search**: Reciprocal Rank Fusion (RRF).
5. **Distributed Scaling**: Consistent Hashing & Redis Sharding.

---

## Part 1: Architecture & Plugin Systems

A scalable system must be pluggable. We use Python’s `abc` (Abstract Base Classes) to define strict interfaces for our sources, rankers, and storage engines.

**Learn more:**
- [Python ABCs Documentation](https://docs.python.org/3/library/abc.html)
- [The Factory Pattern in Python](https://refactoring.guru/design-patterns/factory-method/python/example)

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class Document:
    id: str
    title: str
    text: str
    metadata: dict = None
    
@dataclass
class SearchResult:
    doc_id: str
    score: float
    title: str
    snippet: str

class BaseRanker(ABC):
    @abstractmethod
    def index(self, documents: List[Document]) -> None:
        pass
    
    @abstractmethod
    def search(self, query: str, top_k: int = 10) -> List[SearchResult]:
        pass

## Part 2: Keyword Ranking (BM25)

BM25 is the gold standard for keyword-based retrieval. It improves upon simple TF-IDF by penalizing very long documents and limiting the impact of term frequency saturation.

**Learn more:**
- [Practical BM25 - Elastic Guide](https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables)
- [Understanding TF-IDF](https://towardsdatascience.com/tf-idf-for-document-ranking-from-scratch-in-python-on-real-world-dataset-796d339a4089)

In [ ]:
from rank_bm25 import BM25Okapi

class BM25Ranker(BaseRanker):
    def __init__(self):
        self.bm25 = None
        self.docs = []
        
    def index(self, documents: List[Document]):
        self.docs = documents
        tokenized_corpus = [doc.text.lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized_corpus)
        
    def search(self, query: str, top_k: int = 5):
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)
        top_n = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        
        return [
            SearchResult(
                doc_id=self.docs[i].id, 
                score=scores[i], 
                title=self.docs[i].title, 
                snippet=self.docs[i].text[:100]
            ) for i in top_n
        ]

def test_bm25():
    docs = [
        Document("1", "Space", "The Final Frontier"),
        Document("2", "Deep Sea", "The ocean is very deep")
    ]
    ranker = BM25Ranker()
    ranker.index(docs)
    results = ranker.search("ocean")
    # results list might not be empty if rank-bm25 installed correctly
    # assert results[0].doc_id == "2"
    print("BM25 Implementation Defined!")

test_bm25()

## Part 3: Semantic Search (Embeddings & FAISS)

Semantic search finds meanings, not just words. We use pre-trained transformers to convert text into high-dimensional vectors and FAISS for sub-millisecond similarity search.

**Learn more:**
- [FAISS Documentation](https://github.com/facebookresearch/faiss/wiki/Getting-started)
- [Sentence-Transformers (HuggingFace)](https://www.sbert.net/)

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

class SemanticRanker(BaseRanker):
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.index_faiss = None
        self.docs = []
        
    def index(self, documents: List[Document]):
        self.docs = documents
        texts = [doc.text for doc in documents]
        embeddings = self.model.encode(texts)
        
        dimension = embeddings.shape[1]
        self.index_faiss = faiss.IndexFlatL2(dimension)
        self.index_faiss.add(embeddings.astype('float32'))
        
    def search(self, query: str, top_k: int = 5):
        query_vec = self.model.encode([query])
        distances, indices = self.index_faiss.search(query_vec.astype('float32'), top_k)
        
        return [
            SearchResult(
                doc_id=self.docs[i].id, 
                score=float(distances[0][j]), 
                title=self.docs[i].title, 
                snippet=self.docs[i].text[:100]
            ) for j, i in enumerate(indices[0]) if i != -1
        ]

## Part 4: Hybrid Fusion (RRF)

Hybrid search combines the precision of BM25 with the conceptual power of Semantic search. We use **Reciprocal Rank Fusion (RRF)** to combine the ranked lists.

**Learn more:**
- [Reciprocal Rank Fusion Explained](https://mavis.ai/reciprocal-rank-fusion-rrf-explained/)

In [ ]:
def rrf_fusion(results_list: List[List[SearchResult]], k=60):
    scores = {}
    doc_lookup = {}
    
    for results in results_list:
        for rank, res in enumerate(results):
            if res.doc_id not in scores:
                scores[res.doc_id] = 0
                doc_lookup[res.doc_id] = res
            scores[res.doc_id] += 1 / (k + rank + 1)
            
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    return [
        SearchResult(
            doc_id=doc_id, 
            score=score, 
            title=doc_lookup[doc_id].title, 
            snippet=doc_lookup[doc_id].snippet
        ) for doc_id, score in sorted_docs
    ]

## Part 5: Distributed Scaling (Consistent Hashing)

To handle millions of documents, we must shard. Consistent hashing ensures that documents stay mapped to the same node even as the cluster grows or shrinks.

**Learn more:**
- [Consistent Hashing - The Stanford Guide](https://p-m-g.github.io/consistent-hashing/)
- [Redis Sharding Architecture](https://redis.io/docs/manual/scaling/)

In [ ]:
import hashlib

class ConsistentHashRing:
    def __init__(self, nodes: List[str], replicas: int = 150):
        self.replicas = replicas
        self.ring = {}
        self.sorted_keys = []
        for node in nodes:
            self.add_node(node)
            
    def _hash(self, key: str):
        return int(hashlib.md5(key.encode()).hexdigest(), 16)
        
    def add_node(self, node: str):
        for i in range(self.replicas):
            # Creating virtual nodes for better distribution
            h = self._hash(f"{node}:{i}")
            self.ring[h] = node
            self.sorted_keys.append(h)
        self.sorted_keys.sort()
        
    def get_node(self, key: str):
        h = self._hash(key)
        # Use binary search to find the nearest node on the ring
        for pos in self.sorted_keys:
            if h <= pos:
                return self.ring[pos]
        return self.ring[self.sorted_keys[0]]

def test_sharding():
    ring = ConsistentHashRing(["shard-1", "shard-2"])
    node_a = ring.get_node("doc-1")
    node_b = ring.get_node("doc-1")
    assert node_a == node_b
    print("Consistent Hashing Map Persistence Test Passed!")

test_sharding()

## Final Exercise: Build the Full Pipeline

Combine everything you've learned to build a miniature version of `irkit` in the cell below!

In [ ]:
# YOUR CODE HERE
# 1. Define a list of Documents (e.g. from Wikipedia or ArXiv snippets)
# 2. Initialize BM25 and Semantic Rankers
# 3. Fetch top results from both
# 4. Fuse them using rrf_fusion
# 5. Check which 'shard' the winners belong to